# Fine-tuning Qwen3-4B for Recursive Language Model (RLM)

Based on: **"Recursive Language Models"** (Zhang et al., 2025) - arXiv:2512.24601

This notebook fine-tunes Qwen3-4B-Instruct-2507 with:
1. **Coding datasets** - Build strong code generation skills
2. **RLM trajectories** - Learn REPL interaction patterns

**Run in Google Colab with T4/A100 GPU**

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate peft trl bitsandbytes datasets huggingface_hub

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. RLM System Prompt (from Paper Appendix D.1)

In [ ]:
RLM_SYSTEM_PROMPT = '''You are tasked with answering a query with associated context. You can access, transform, and analyze this context interactively in a REPL environment that can recursively query sub-LLMs.

Your context is a {context_type} with {context_total_length} total characters.

The REPL environment is initialized with:
1. A 'context' variable containing the input data
2. A 'llm_query' function to recursively call sub-LLMs
3. The ability to use 'print()' to view outputs

IMPORTANT: Batch llm_query calls efficiently (~200k chars per call).

Write Python code in ```repl blocks. When done, use:
- FINAL(your answer) for direct answers
- FINAL_VAR(variable_name) to return a variable

Think step by step and execute your plan immediately.'''

print(f"System prompt: {len(RLM_SYSTEM_PROMPT)} chars")

---
## 2. Load Coding Datasets

We combine multiple coding datasets:
- **Evol-Instruct-Code** - WizardCoder's synthetic code instructions
- **CodeSearchNet** - Functions with docstrings
- **MBPP** - Python coding problems
- **Custom RLM trajectories** - REPL interaction patterns

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
import random
import json

# ============================================================================
# 2.1 Evol-Instruct-Code (WizardCoder style)
# ============================================================================
print("Loading Evol-Instruct-Code...")
try:
    evol_code = load_dataset("WizardLMTeam/WizardLM_evol_instruct_V2_196k", split="train")
    # Filter for code-related
    code_keywords = ['code', 'function', 'python', 'program', 'algorithm', 'implement', 'write a', 'def ', 'class ']
    evol_code = evol_code.filter(
        lambda x: any(kw in x['conversations'][0]['value'].lower() for kw in code_keywords)
    )
    evol_code = evol_code.shuffle(seed=42).select(range(min(5000, len(evol_code))))
    print(f"  Evol-Instruct-Code: {len(evol_code)} examples")
except Exception as e:
    print(f"  Could not load Evol-Instruct: {e}")
    evol_code = None

In [ ]:
# ============================================================================
# 2.2 CodeSearchNet (Python functions with docstrings)
# ============================================================================
print("Loading CodeSearchNet...")
try:
    codesearch = load_dataset("code_search_net", "python", split="train", trust_remote_code=True)
    codesearch = codesearch.shuffle(seed=42).select(range(min(5000, len(codesearch))))
    print(f"  CodeSearchNet: {len(codesearch)} examples")
except Exception as e:
    print(f"  Could not load CodeSearchNet: {e}")
    codesearch = None

In [ ]:
# ============================================================================
# 2.3 MBPP (Mostly Basic Python Problems)
# ============================================================================
print("Loading MBPP...")
try:
    mbpp = load_dataset("mbpp", split="train", trust_remote_code=True)
    print(f"  MBPP: {len(mbpp)} examples")
except Exception as e:
    print(f"  Could not load MBPP: {e}")
    mbpp = None

In [ ]:
# ============================================================================
# 2.4 OpenHermes Code subset (high quality)
# ============================================================================
print("Loading OpenHermes...")
try:
    hermes = load_dataset("teknium/OpenHermes-2.5", split="train")
    # Filter for code
    hermes = hermes.filter(
        lambda x: any(kw in str(x.get('conversations', [{}])[0].get('value', '')).lower() 
                     for kw in ['```python', '```', 'def ', 'function', 'code'])
    )
    hermes = hermes.shuffle(seed=42).select(range(min(3000, len(hermes))))
    print(f"  OpenHermes (code): {len(hermes)} examples")
except Exception as e:
    print(f"  Could not load OpenHermes: {e}")
    hermes = None

---
## 3. Generate RLM Training Examples

Create synthetic RLM trajectories that teach the model to:
- Examine context with code
- Use `llm_query()` for recursive calls
- Output `FINAL()` / `FINAL_VAR()`

In [ ]:
def generate_rlm_needle_example():
    """Needle-in-haystack: Find hidden value in large context."""
    needle_value = random.randint(1000, 9999)
    patterns = [
        (f"SECRET_CODE={needle_value}", "SECRET_CODE"),
        (f"MAGIC_NUMBER: {needle_value}", "MAGIC_NUMBER"),
        (f"hidden_key = {needle_value}", "hidden_key"),
        (f"# PASSWORD: {needle_value}", "PASSWORD"),
    ]
    needle, name = random.choice(patterns)
    
    return {
        "messages": [
            {"role": "system", "content": RLM_SYSTEM_PROMPT.format(
                context_type="string", context_total_length="~100,000"
            )},
            {"role": "user", "content": f"Find the {name} value in this document."},
            {"role": "assistant", "content": f'''Let me search the context for {name}.

```repl
print(f"Context length: {{len(context)}} chars")
print(f"Preview: {{context[:200]}}")
```

Now searching for the pattern:

```repl
import re
pattern = r"{name}[=:\\s]+([\\d]+)"
match = re.search(pattern, context)
if match:
    result = match.group(1)
    print(f"Found: {{result}}")
else:
    # Search in chunks
    for i in range(0, len(context), 20000):
        chunk = context[i:i+20000]
        if "{name}" in chunk:
            result = llm_query(f"Extract the {name} value from: {{chunk}}")
            print(f"Found in chunk {{i//20000}}: {{result}}")
            break
```

FINAL({needle_value})'''}
        ]
    }


def generate_rlm_aggregation_example():
    """Aggregation: Count/classify items across large dataset."""
    categories = ["error", "warning", "info", "debug"]
    target = random.choice(categories)
    
    return {
        "messages": [
            {"role": "system", "content": RLM_SYSTEM_PROMPT.format(
                context_type="log file", context_total_length="~500,000"
            )},
            {"role": "user", "content": f"Count all '{target}' level log entries."},
            {"role": "assistant", "content": f'''I'll count {target} entries in the logs.

```repl
lines = context.strip().split('\\n')
print(f"Total lines: {{len(lines)}}")
print(f"Sample: {{lines[:3]}}")
```

Now counting:

```repl
count = 0
for line in lines:
    if '[{target.upper()}]' in line or '{target}' in line.lower():
        count += 1

print(f"Total {target} entries: {{count}}")
result = count
```

FINAL_VAR(result)'''}
        ]
    }


def generate_rlm_code_search_example():
    """Code search: Find functions/classes in codebase."""
    targets = [
        ("authentication", ["login", "logout", "auth", "verify_token"]),
        ("database", ["query", "connect", "execute", "fetch"]),
        ("validation", ["validate", "check", "verify", "sanitize"]),
        ("API endpoints", ["@app.route", "@router", "def get_", "def post_"]),
    ]
    topic, keywords = random.choice(targets)
    
    return {
        "messages": [
            {"role": "system", "content": RLM_SYSTEM_PROMPT.format(
                context_type="list of code files", context_total_length="~2,000,000"
            )},
            {"role": "user", "content": f"Find all functions related to {topic} and explain what each does."},
            {"role": "assistant", "content": f'''I'll search the codebase for {topic}-related functions.

```repl
print(f"Files in codebase: {{len(context)}}")
print(f"File structure: {{type(context[0]) if context else 'empty'}}")
```

Searching for relevant patterns:

```repl
import re
keywords = {keywords}
found_functions = []

for file_info in context:
    filename = file_info.get('filename', '')
    content = file_info.get('content', '')
    
    # Find function definitions
    for match in re.finditer(r'def\\s+(\\w+)\\s*\\([^)]*\\):', content):
        func_name = match.group(1)
        if any(kw in func_name.lower() for kw in keywords):
            # Get function body (next 10 lines)
            start = match.start()
            snippet = content[start:start+500]
            found_functions.append({{
                'name': func_name,
                'file': filename,
                'code': snippet
            }})

print(f"Found {{len(found_functions)}} functions")
for f in found_functions[:5]:
    print(f"- {{f['name']}} in {{f['file']}}")
```

Now let me explain each function:

```repl
explanations = []
for func in found_functions:
    explanation = llm_query(f"""Explain this function in 1 sentence:
File: {{func['file']}}
{{func['code']}}""")
    explanations.append(f"- {{func['name']}}: {{explanation}}")

result = '\\n'.join(explanations)
print(result)
```

FINAL_VAR(result)'''}
        ]
    }


def generate_rlm_multi_doc_example():
    """Multi-document QA: Answer questions spanning many documents."""
    questions = [
        "Which documents discuss both performance optimization and security?",
        "Find all documents that reference the main API and summarize their purpose.",
        "List documents containing TODO comments and what needs to be done.",
        "Which files import the 'utils' module and what do they use it for?",
    ]
    question = random.choice(questions)
    
    return {
        "messages": [
            {"role": "system", "content": RLM_SYSTEM_PROMPT.format(
                context_type="list of 500 documents", context_total_length="~5,000,000"
            )},
            {"role": "user", "content": question},
            {"role": "assistant", "content": f'''I'll search through all documents to answer this.

```repl
print(f"Total documents: {{len(context)}}")
print(f"Sample doc keys: {{context[0].keys() if context else 'none'}}")
```

Searching documents:

```repl
matching_docs = []

# Process in batches to use llm_query efficiently
batch_size = 20
for i in range(0, len(context), batch_size):
    batch = context[i:i+batch_size]
    batch_text = '\\n---\\n'.join(
        f"FILE: {{d.get('filename', f'doc_{{j}}')}}}\\n{{d.get('content', '')[:2000]}}"
        for j, d in enumerate(batch)
    )
    
    result = llm_query(f"""{question}
    
Documents:
{{batch_text}}

List matching filenames only, one per line, or 'NONE' if no matches.""")
    
    if result.strip() != 'NONE':
        matching_docs.extend(result.strip().split('\\n'))
    
    if (i // batch_size) % 5 == 0:
        print(f"Processed {{i + batch_size}}/{{len(context)}} docs, found {{len(matching_docs)}} matches")

print(f"\\nTotal matches: {{len(matching_docs)}}")
result = matching_docs
```

FINAL_VAR(result)'''}
        ]
    }


# Generate RLM examples
print("Generating RLM training examples...")
rlm_generators = [
    generate_rlm_needle_example,
    generate_rlm_aggregation_example,
    generate_rlm_code_search_example,
    generate_rlm_multi_doc_example,
]

rlm_examples = []
for i in range(2000):  # 2000 RLM examples
    gen = random.choice(rlm_generators)
    rlm_examples.append(gen())

print(f"Generated {len(rlm_examples)} RLM examples")

---
## 4. Format All Datasets

In [ ]:
def format_evol_instruct(example):
    """Format WizardLM/Evol-Instruct to chat format."""
    convs = example.get('conversations', [])
    if len(convs) < 2:
        return None
    return {
        "messages": [
            {"role": "system", "content": "You are an expert Python programmer. Write clean, efficient code."},
            {"role": "user", "content": convs[0].get('value', '')},
            {"role": "assistant", "content": convs[1].get('value', '')}
        ]
    }

def format_codesearchnet(example):
    """Format CodeSearchNet to chat format."""
    docstring = example.get('func_documentation_string', '')
    code = example.get('func_code_string', '')
    if not docstring or not code:
        return None
    return {
        "messages": [
            {"role": "system", "content": "You are an expert Python programmer."},
            {"role": "user", "content": f"Write a Python function that: {docstring}"},
            {"role": "assistant", "content": f"```python\n{code}\n```"}
        ]
    }

def format_mbpp(example):
    """Format MBPP to chat format."""
    return {
        "messages": [
            {"role": "system", "content": "You are an expert Python programmer. Write clean, tested code."},
            {"role": "user", "content": example.get('text', '')},
            {"role": "assistant", "content": f"```python\n{example.get('code', '')}\n```"}
        ]
    }

def format_hermes(example):
    """Format OpenHermes to chat format."""
    convs = example.get('conversations', [])
    if len(convs) < 2:
        return None
    return {
        "messages": [
            {"role": "system", "content": "You are an expert programmer and helpful assistant."},
            {"role": "user", "content": convs[0].get('value', '')},
            {"role": "assistant", "content": convs[1].get('value', '')}
        ]
    }

# Format all datasets
all_examples = []

if evol_code:
    for ex in evol_code:
        formatted = format_evol_instruct(ex)
        if formatted:
            all_examples.append(formatted)
    print(f"Added {len([e for e in evol_code])} Evol-Instruct examples")

if codesearch:
    for ex in codesearch:
        formatted = format_codesearchnet(ex)
        if formatted:
            all_examples.append(formatted)
    print(f"Added CodeSearchNet examples")

if mbpp:
    for ex in mbpp:
        formatted = format_mbpp(ex)
        if formatted:
            all_examples.append(formatted)
    print(f"Added MBPP examples")

if hermes:
    for ex in hermes:
        formatted = format_hermes(ex)
        if formatted:
            all_examples.append(formatted)
    print(f"Added OpenHermes examples")

# Add RLM examples
all_examples.extend(rlm_examples)
print(f"Added {len(rlm_examples)} RLM examples")

# Shuffle
random.shuffle(all_examples)

print(f"\n=== Total training examples: {len(all_examples)} ===")

In [ ]:
# Create HuggingFace dataset
from datasets import Dataset

train_dataset = Dataset.from_list(all_examples)
print(f"Dataset size: {len(train_dataset)}")
print(f"Sample: {train_dataset[0]['messages'][1]['content'][:200]}...")

---
## 5. Load Model with QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# QLoRA config - 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",  # For compatibility
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded!")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Vocab size: {model.config.vocab_size}")

In [ ]:
# LoRA config - target all important layers
lora_config = LoraConfig(
    r=64,                     # Rank (higher = more capacity)
    lora_alpha=128,           # Alpha scaling
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[          # Qwen3 modules
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj"       # MLP
    ],
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

---
## 6. Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Format function for chat template
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

# Training config - use TrainingArguments for compatibility
training_args = TrainingArguments(
    output_dir="./rlm-qwen3-4b-coder",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=25,
    save_steps=500,
    save_total_limit=3,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

MAX_SEQ_LENGTH = 4096

print("Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    processing_class=tokenizer,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"\nStarting training on {len(train_dataset)} examples...")
print("This will take ~2-3 hours on T4, ~45 min on A100\n")

trainer.train()

In [ ]:
# Save the LoRA adapter
trainer.save_model("./rlm-qwen3-4b-coder-final")
tokenizer.save_pretrained("./rlm-qwen3-4b-coder-final")
print("LoRA adapter saved!")

---
## 7. Test the Fine-tuned Model

In [ ]:
# Test 1: Pure coding task
print("=" * 60)
print("TEST 1: Coding Task")
print("=" * 60)

messages = [
    {"role": "system", "content": "You are an expert Python programmer."},
    {"role": "user", "content": "Write a function to find all prime numbers up to n using the Sieve of Eratosthenes."}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
# Test 2: RLM task
print("\n" + "=" * 60)
print("TEST 2: RLM Task (should write REPL code)")
print("=" * 60)

messages = [
    {"role": "system", "content": RLM_SYSTEM_PROMPT.format(
        context_type="string", context_total_length="500,000"
    )},
    {"role": "user", "content": "Find the SECRET_KEY value hidden somewhere in this document."}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
# Test 3: Code search RLM task
print("\n" + "=" * 60)
print("TEST 3: Code Search RLM Task")
print("=" * 60)

messages = [
    {"role": "system", "content": RLM_SYSTEM_PROMPT.format(
        context_type="list of 200 Python files", context_total_length="3,000,000"
    )},
    {"role": "user", "content": "Find all functions that handle user authentication and explain what each does."}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

---
## 8. Merge and Export

In [ ]:
# Merge LoRA into base model for deployment
from peft import PeftModel

print("Loading base model for merging...")

# Load base model in full precision
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Load and merge LoRA
print("Merging LoRA weights...")
model = PeftModel.from_pretrained(base_model, "./rlm-qwen3-4b-coder-final")
merged_model = model.merge_and_unload()

# Save
print("Saving merged model...")
merged_model.save_pretrained("./rlm-qwen3-4b-merged")
tokenizer.save_pretrained("./rlm-qwen3-4b-merged")

print("\nMerged model saved to ./rlm-qwen3-4b-merged")
print("You can now upload this to HuggingFace or use it directly!")

In [ ]:
# Optional: Upload to HuggingFace
from huggingface_hub import login, HfApi

# login()  # Uncomment and run to login

# HF_USERNAME = "your-username"
# merged_model.push_to_hub(f"{HF_USERNAME}/rlm-qwen3-4b-coder")
# tokenizer.push_to_hub(f"{HF_USERNAME}/rlm-qwen3-4b-coder")
# print(f"Uploaded to https://huggingface.co/{HF_USERNAME}/rlm-qwen3-4b-coder")

---
## 9. Full RLM Inference Test

Now let's test the model with a **real RLM loop** - executing REPL code and handling sub-LLM calls!

In [ ]:
import re
from typing import Any, List, Tuple

class REPLEnvironment:
    """Python REPL for RLM execution."""
    
    def __init__(self, context: Any, llm_query_fn):
        self.output_buffer = []
        
        def captured_print(*args):
            self.output_buffer.append(' '.join(str(a) for a in args))
        
        self.namespace = {
            'context': context,
            'llm_query': llm_query_fn,
            'print': captured_print,
            'len': len, 'range': range, 'enumerate': enumerate,
            'str': str, 'int': int, 'float': float, 'list': list, 'dict': dict,
            'sum': sum, 'min': min, 'max': max, 'sorted': sorted,
            're': __import__('re'),
            'json': __import__('json'),
        }
    
    def execute(self, code: str) -> Tuple[bool, str]:
        self.output_buffer = []
        try:
            exec(code, self.namespace)
            output = '\n'.join(self.output_buffer)
            return True, output if output else '[OK]'
        except Exception as e:
            return False, f'Error: {e}'
    
    def get_var(self, name: str):
        return self.namespace.get(name)


def extract_repl_code(response: str) -> List[str]:
    """Extract ```repl code blocks."""
    return re.findall(r'```repl\s*\n(.*?)```', response, re.DOTALL)


def extract_final(response: str, repl: REPLEnvironment):
    """Extract FINAL() or FINAL_VAR()."""
    var_match = re.search(r'FINAL_VAR\((\w+)\)', response)
    if var_match:
        return str(repl.get_var(var_match.group(1)))
    
    final_match = re.search(r'FINAL\(([^)]+)\)', response)
    if final_match:
        return final_match.group(1).strip()
    
    return None


print("RLM inference helpers loaded!")

In [ ]:
def run_rlm_inference(
    model, 
    tokenizer, 
    context: Any, 
    question: str,
    max_iterations: int = 10,
    verbose: bool = True
):
    """
    Run full RLM inference loop.
    
    1. Send question to model
    2. Extract and execute ```repl code
    3. Feed results back to model
    4. Repeat until FINAL() is produced
    """
    
    # Sub-LLM query function
    sub_call_count = [0]
    def llm_query(prompt: str) -> str:
        sub_call_count[0] += 1
        if verbose:
            print(f"  [Sub-LLM call #{sub_call_count[0]}]")
        
        msgs = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
        
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Create REPL
    repl = REPLEnvironment(context, llm_query)
    
    # System prompt
    context_len = len(context) if isinstance(context, str) else len(str(context))
    system = RLM_SYSTEM_PROMPT.format(
        context_type=type(context).__name__,
        context_total_length=f"{context_len:,}"
    )
    
    # Conversation
    conversation = [question]
    
    for i in range(max_iterations):
        if verbose:
            print(f"\n{'='*50}")
            print(f"Iteration {i+1}")
            print('='*50)
        
        # Generate response
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": '\n\n'.join(conversation)}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=2048, 
                temperature=0.7, 
                do_sample=True
            )
        
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        if verbose:
            print(f"\nModel response:\n{response[:500]}{'...' if len(response) > 500 else ''}")
        
        # Check for final answer
        final = extract_final(response, repl)
        if final:
            if verbose:
                print(f"\n{'='*50}")
                print(f"FINAL ANSWER: {final}")
                print(f"Sub-LLM calls: {sub_call_count[0]}")
                print('='*50)
            return final
        
        # Execute REPL code
        code_blocks = extract_repl_code(response)
        
        if not code_blocks:
            conversation.append(response)
            conversation.append("[No REPL code found. Write ```repl code or provide FINAL(answer).]")
            continue
        
        # Execute each block
        exec_results = []
        for j, code in enumerate(code_blocks):
            if verbose:
                print(f"\nExecuting code block {j+1}...")
            success, output = repl.execute(code)
            exec_results.append(f"[Block {j+1}]: {output[:500]}")
            if verbose:
                print(f"Output: {output[:200]}{'...' if len(output) > 200 else ''}")
        
        conversation.append(response)
        conversation.append('\n'.join(exec_results))
    
    return "[Max iterations reached]"


print("RLM inference function ready!")

In [ ]:
# =============================================================================
# TEST: Needle in Haystack
# =============================================================================
import random

print("="*60)
print("TEST: Needle in Haystack")
print("="*60)

# Generate haystack with hidden needle
words = ['lorem', 'ipsum', 'dolor', 'sit', 'amet', 'code', 'function', 'data']
haystack_parts = []
for i in range(500):
    line = ' '.join(random.choices(words, k=10))
    haystack_parts.append(line)

# Insert needle at random position
needle_pos = random.randint(200, 400)
secret_value = random.randint(1000, 9999)
haystack_parts.insert(needle_pos, f"SECRET_KEY={secret_value}")

test_context = '\n'.join(haystack_parts)
print(f"Context size: {len(test_context):,} chars")
print(f"Hidden SECRET_KEY={secret_value} at line {needle_pos}")

# Run RLM
answer = run_rlm_inference(
    model, 
    tokenizer,
    context=test_context,
    question="Find the SECRET_KEY value hidden in this document.",
    verbose=True
)

print(f"\nExpected: {secret_value}")
print(f"Got: {answer}")
print(f"Correct: {str(secret_value) in str(answer)}")

In [ ]:
# =============================================================================
# TEST: Aggregation (Count items)
# =============================================================================
print("\n" + "="*60)
print("TEST: Aggregation")
print("="*60)

# Generate log-like data
log_levels = ['INFO', 'WARNING', 'ERROR', 'DEBUG']
log_lines = []
level_counts = {level: 0 for level in log_levels}

for i in range(200):
    level = random.choice(log_levels)
    level_counts[level] += 1
    log_lines.append(f"[{level}] 2025-01-{random.randint(1,28):02d} Message {i}")

test_context = '\n'.join(log_lines)
target_level = 'ERROR'
expected_count = level_counts[target_level]

print(f"Context: {len(log_lines)} log lines")
print(f"Level counts: {level_counts}")

answer = run_rlm_inference(
    model,
    tokenizer,
    context=test_context,
    question=f"Count how many {target_level} level log entries are in this log file.",
    verbose=True
)

print(f"\nExpected: {expected_count}")
print(f"Got: {answer}")

In [ ]:
# =============================================================================
# TEST: Multi-file Code Search
# =============================================================================
print("\n" + "="*60)
print("TEST: Code Search")
print("="*60)

# Simulate codebase
fake_codebase = [
    {
        'filename': 'auth/login.py',
        'content': '''def login_user(username, password):
    """Authenticate user with username and password."""
    user = db.get_user(username)
    if user and verify_password(password, user.hash):
        return create_session(user)
    return None

def logout_user(session_id):
    """End user session."""
    return db.delete_session(session_id)
'''
    },
    {
        'filename': 'auth/tokens.py', 
        'content': '''def verify_token(token):
    """Verify JWT token validity."""
    try:
        payload = jwt.decode(token, SECRET_KEY)
        return payload
    except jwt.InvalidTokenError:
        return None
'''
    },
    {
        'filename': 'api/users.py',
        'content': '''def get_users():
    """Get all users."""
    return db.query(User).all()

def create_user(data):
    """Create new user."""
    user = User(**data)
    db.add(user)
    return user
'''
    },
    {
        'filename': 'utils/helpers.py',
        'content': '''def format_date(dt):
    return dt.strftime("%Y-%m-%d")

def validate_email(email):
    import re
    return bool(re.match(r"[^@]+@[^@]+\\.[^@]+", email))
'''
    }
]

print(f"Codebase: {len(fake_codebase)} files")

answer = run_rlm_inference(
    model,
    tokenizer,
    context=fake_codebase,
    question="Find all functions related to authentication and list them with their file paths.",
    verbose=True
)

print(f"\nAnswer:\n{answer}")

In [ ]:
# =============================================================================
# Interactive Test - Try Your Own!
# =============================================================================
print("\n" + "="*60)
print("INTERACTIVE: Try your own RLM query!")
print("="*60)

# Put your own context here
my_context = """
This is a sample document. You can replace this with:
- A long text file
- A list of documents
- Code files
- Log data
- Anything you want the model to analyze!

The RLM will write Python code to examine this context
and recursively call sub-LLMs if needed.

Hidden secret: ANSWER=42
"""

my_question = "What is the hidden ANSWER value?"

answer = run_rlm_inference(
    model,
    tokenizer,
    context=my_context,
    question=my_question,
    verbose=True
)

print(f"\nFinal Answer: {answer}")

---
## 10. Real Benchmarks

Evaluate on standard coding and long-context benchmarks:
- **HumanEval**: Code generation (pass@1)
- **MBPP**: Python programming problems
- **S-NIAH (RULER)**: Single needle-in-a-haystack
- **RULER Multi-Key**: Multiple key retrieval

In [ ]:
# =============================================================================
# BENCHMARK 1: HumanEval (Code Generation)
# =============================================================================
print("="*60)
print("BENCHMARK: HumanEval (subset)")
print("="*60)

# Sample HumanEval problems
humaneval_problems = [
    {
        "task_id": "HumanEval/0",
        "prompt": "from typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    \"\"\" Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    \"\"\"",
        "canonical_solution": "    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = abs(elem - elem2)\n                if distance < threshold:\n                    return True\n    return False\n",
        "test": "def check(candidate):\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False\n    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0], 2.0) == True\n    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 1.0) == True\n    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 0.5) == False\n"
    },
    {
        "task_id": "HumanEval/1",
        "prompt": "from typing import List\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    \"\"\" Input to this function is a string containing multiple groups of nested parentheses. Your goal is to\n    separate those group into separate strings and return the list of those.\n    Separate groups are balanced (each open brace is properly closed) and not nested within each other\n    Ignore any spaces in the input string.\n    >>> separate_paren_groups('( ) (( )) (( )( ))')\n    ['()', '(())', '(()())']\n    \"\"\"",
        "canonical_solution": "    result = []\n    current_string = []\n    current_depth = 0\n\n    for c in paren_string:\n        if c == '(':\n            current_depth += 1\n            current_string.append(c)\n        elif c == ')':\n            current_depth -= 1\n            current_string.append(c)\n\n            if current_depth == 0:\n                result.append(''.join(current_string))\n                current_string = []\n\n    return result\n",
        "test": "def check(candidate):\n    assert candidate('(()()) ((())) () ((())()())') == ['(()())', '((()))', '()', '((())()())']\n    assert candidate('() (()) ((())) (((())))') == ['()', '(())', '((()))', '(((())))']\n    assert candidate('(()(()))') == ['(()(()))']\n"
    },
    {
        "task_id": "HumanEval/4",
        "prompt": "from typing import List\n\ndef mean_absolute_deviation(numbers: List[float]) -> float:\n    \"\"\" For a given list of input numbers, calculate Mean Absolute Deviation\n    around the mean of this dataset.\n    Mean Absolute Deviation is the average absolute difference between each\n    element and a centerpoint (mean in this case):\n    MAD = average | x - x_mean |\n    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])\n    1.0\n    \"\"\"",
        "canonical_solution": "    mean = sum(numbers) / len(numbers)\n    return sum(abs(x - mean) for x in numbers) / len(numbers)\n",
        "test": "def check(candidate):\n    assert abs(candidate([1.0, 2.0, 3.0]) - 2.0/3.0) < 1e-6\n    assert abs(candidate([1.0, 2.0, 3.0, 4.0]) - 1.0) < 1e-6\n    assert abs(candidate([1.0, 2.0, 3.0, 4.0, 5.0]) - 6.0/5.0) < 1e-6\n"
    }
]

def run_humaneval_test(model, tokenizer, problem):
    """Run a single HumanEval problem."""
    messages = [
        {"role": "system", "content": "You are an expert Python programmer. Complete the function. Output ONLY the function body, no explanations."},
        {"role": "user", "content": f"Complete this function:\n\n{problem['prompt']}"}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.2, do_sample=True)
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Extract code
    if '```python' in response:
        code = response.split('```python')[1].split('```')[0]
    elif '```' in response:
        code = response.split('```')[1].split('```')[0]
    else:
        code = response
    
    # Try to run test
    try:
        full_code = problem['prompt'] + code + '\n' + problem['test'] + f"\ncheck({problem['prompt'].split('def ')[1].split('(')[0]})"
        exec(full_code, {})
        return True, code
    except Exception as e:
        return False, str(e)

# Run HumanEval subset
humaneval_results = []
for prob in humaneval_problems:
    passed, result = run_humaneval_test(model, tokenizer, prob)
    humaneval_results.append(passed)
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f"{prob['task_id']}: {status}")

humaneval_score = sum(humaneval_results) / len(humaneval_results) * 100
print(f"\nHumanEval (subset): {humaneval_score:.1f}% ({sum(humaneval_results)}/{len(humaneval_results)})")

In [ ]:
# =============================================================================
# BENCHMARK 2: MBPP (Mostly Basic Python Problems)
# =============================================================================
print("\n" + "="*60)
print("BENCHMARK: MBPP (subset)")
print("="*60)

# Load MBPP test split
try:
    mbpp_test = load_dataset("mbpp", split="test", trust_remote_code=True)
    mbpp_test = mbpp_test.select(range(min(20, len(mbpp_test))))  # Test on 20 samples
    
    def run_mbpp_test(model, tokenizer, example):
        """Run a single MBPP problem."""
        messages = [
            {"role": "system", "content": "You are an expert Python programmer. Write a function to solve the problem. Output ONLY the Python code."},
            {"role": "user", "content": example['text']}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.2, do_sample=True)
        
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Extract code
        if '```python' in response:
            code = response.split('```python')[1].split('```')[0]
        elif '```' in response:
            code = response.split('```')[1].split('```')[0]
        else:
            code = response
        
        # Run tests
        try:
            test_code = code + '\n' + '\n'.join(example['test_list'])
            exec(test_code, {})
            return True
        except:
            return False
    
    mbpp_results = []
    for i, ex in enumerate(mbpp_test):
        passed = run_mbpp_test(model, tokenizer, ex)
        mbpp_results.append(passed)
        if (i+1) % 5 == 0:
            print(f"Progress: {i+1}/{len(mbpp_test)} - Running accuracy: {sum(mbpp_results)/len(mbpp_results)*100:.1f}%")
    
    mbpp_score = sum(mbpp_results) / len(mbpp_results) * 100
    print(f"\nMBPP (subset): {mbpp_score:.1f}% ({sum(mbpp_results)}/{len(mbpp_results)})")
    
except Exception as e:
    print(f"Could not run MBPP benchmark: {e}")
    mbpp_score = None

In [ ]:
# =============================================================================
# BENCHMARK 3: S-NIAH (Single Needle in a Haystack) - from RULER
# =============================================================================
print("\n" + "="*60)
print("BENCHMARK: S-NIAH (RULER style)")
print("="*60)

def generate_ruler_niah(context_length: int, needle_depth: float = 0.5):
    """
    Generate RULER-style needle-in-a-haystack test.
    
    Args:
        context_length: Target context length in chars
        needle_depth: Where to place needle (0.0 = start, 1.0 = end)
    """
    # The needle
    magic_number = random.randint(10000, 99999)
    needle = f"The special magic number is {magic_number}."
    
    # Generate haystack (Paul Graham essays style filler)
    filler_sentences = [
        "The best way to predict the future is to invent it.",
        "Innovation distinguishes between a leader and a follower.",
        "Stay hungry, stay foolish.",
        "The only way to do great work is to love what you do.",
        "Design is not just what it looks like and feels like.",
        "Design is how it works.",
        "Simple can be harder than complex.",
        "You have to work hard to get your thinking clean.",
        "The people who are crazy enough to think they can change the world are the ones who do.",
        "Quality is more important than quantity.",
        "One home run is much better than two doubles.",
    ]
    
    haystack = []
    current_len = 0
    needle_pos = int(context_length * needle_depth)
    needle_inserted = False
    
    while current_len < context_length:
        if not needle_inserted and current_len >= needle_pos:
            haystack.append(needle)
            current_len += len(needle)
            needle_inserted = True
        else:
            sentence = random.choice(filler_sentences)
            haystack.append(sentence)
            current_len += len(sentence) + 1
    
    context = ' '.join(haystack)
    question = "What is the special magic number mentioned in the text?"
    
    return context, question, str(magic_number)

def run_niah_benchmark(model, tokenizer, context_lengths=[8000, 16000, 32000], n_samples=3):
    """Run S-NIAH at different context lengths."""
    results = {}
    
    for ctx_len in context_lengths:
        print(f"\nTesting context length: {ctx_len:,} chars")
        correct = 0
        
        for i in range(n_samples):
            # Random needle depth
            depth = random.uniform(0.2, 0.8)
            context, question, expected = generate_ruler_niah(ctx_len, depth)
            
            # Run with RLM
            answer = run_rlm_inference(
                model, tokenizer, 
                context=context, 
                question=question,
                verbose=False,
                max_iterations=5
            )
            
            # Check if answer contains the magic number
            is_correct = expected in str(answer)
            correct += int(is_correct)
            
            status = '✓' if is_correct else '✗'
            print(f"  Sample {i+1}: {status} (depth={depth:.2f}, expected={expected}, got={answer[:50]}...)")
        
        accuracy = correct / n_samples * 100
        results[ctx_len] = accuracy
        print(f"  Accuracy at {ctx_len:,}: {accuracy:.1f}%")
    
    return results

# Run S-NIAH benchmark
niah_results = run_niah_benchmark(
    model, tokenizer,
    context_lengths=[8000, 16000, 32000],
    n_samples=3
)

print("\nS-NIAH Results:")
for ctx_len, acc in niah_results.items():
    print(f"  {ctx_len:>6,} chars: {acc:.1f}%")

In [ ]:
# =============================================================================
# BENCHMARK 4: Multi-Key Retrieval (RULER style)
# =============================================================================
print("\n" + "="*60)
print("BENCHMARK: Multi-Key Retrieval")
print("="*60)

def generate_multikey_test(context_length: int, n_keys: int = 5):
    """
    Generate multi-key retrieval test.
    Multiple key-value pairs scattered in text, must retrieve all.
    """
    # Generate key-value pairs
    keys = [f"KEY_{i}" for i in range(n_keys)]
    values = [random.randint(100, 999) for _ in range(n_keys)]
    kv_pairs = list(zip(keys, values))
    
    # Filler text
    filler = [
        "The system processes data efficiently.",
        "Users can access the dashboard anytime.",
        "Performance metrics are tracked daily.",
        "Security protocols are regularly updated.",
        "The API supports multiple endpoints.",
    ]
    
    # Build context with scattered KV pairs
    parts = []
    current_len = 0
    kv_positions = sorted(random.sample(range(10, 90), n_keys))  # Percentages
    kv_idx = 0
    
    while current_len < context_length:
        progress = current_len / context_length * 100
        
        # Insert KV pair at designated position
        if kv_idx < n_keys and progress >= kv_positions[kv_idx]:
            k, v = kv_pairs[kv_idx]
            parts.append(f"Important: {k} = {v}.")
            kv_idx += 1
        else:
            parts.append(random.choice(filler))
        
        current_len = sum(len(p) for p in parts)
    
    context = ' '.join(parts)
    question = f"Find all the KEY values (KEY_0 through KEY_{n_keys-1}) and list them."
    expected = {k: v for k, v in kv_pairs}
    
    return context, question, expected

def run_multikey_benchmark(model, tokenizer, n_samples=3):
    """Run multi-key retrieval benchmark."""
    results = []
    
    for i in range(n_samples):
        n_keys = random.randint(3, 5)
        context, question, expected = generate_multikey_test(20000, n_keys)
        
        print(f"\nSample {i+1}: {n_keys} keys to find")
        
        # Run with RLM
        answer = run_rlm_inference(
            model, tokenizer,
            context=context,
            question=question,
            verbose=False,
            max_iterations=8
        )
        
        # Count how many keys were found correctly
        found = 0
        for k, v in expected.items():
            if str(v) in str(answer):
                found += 1
                print(f"  ✓ Found {k}={v}")
            else:
                print(f"  ✗ Missing {k}={v}")
        
        recall = found / len(expected) * 100
        results.append(recall)
        print(f"  Recall: {recall:.1f}% ({found}/{len(expected)})")
    
    avg_recall = sum(results) / len(results)
    print(f"\nMulti-Key Average Recall: {avg_recall:.1f}%")
    return avg_recall

multikey_score = run_multikey_benchmark(model, tokenizer, n_samples=3)

In [ ]:
# =============================================================================
# BENCHMARK SUMMARY
# =============================================================================
print("\n" + "="*60)
print("BENCHMARK SUMMARY")
print("="*60)

print("\n### Coding Benchmarks ###")
print(f"HumanEval (subset):  {humaneval_score:.1f}%")
if mbpp_score is not None:
    print(f"MBPP (subset):       {mbpp_score:.1f}%")

print("\n### Long-Context Benchmarks (RLM) ###")
print("S-NIAH (needle-in-haystack):")
for ctx_len, acc in niah_results.items():
    print(f"  {ctx_len:>6,} chars: {acc:.1f}%")
print(f"Multi-Key Retrieval: {multikey_score:.1f}%")

print("\n" + "="*60)
print("Note: These are subset evaluations. For full benchmarks, use:")
print("  - bigcode/bigcode-evaluation-harness for HumanEval/MBPP")
print("  - RULER benchmark for comprehensive long-context eval")
print("="*60)

---
## Summary

This notebook fine-tuned **Qwen3-4B-Instruct-2507** with:

### Training Data:
| Dataset | Examples | Purpose |
|---------|----------|----------|
| Evol-Instruct-Code | ~5,000 | Complex coding instructions |
| CodeSearchNet | ~5,000 | Function understanding |
| MBPP | ~400 | Python problems |
| OpenHermes | ~3,000 | General code tasks |
| **RLM Trajectories** | **2,000** | REPL interaction patterns |

### Benchmark Results:
| Benchmark | Score | Notes |
|-----------|-------|-------|
| HumanEval | X% | Code generation |
| MBPP | X% | Python problems |
| S-NIAH 8K | X% | Needle in haystack |
| S-NIAH 32K | X% | Long context |
| Multi-Key | X% | Multiple retrieval |

### The model now knows how to:
1. Write excellent Python code
2. Use ` ```repl ` blocks to examine context
3. Call `llm_query()` for recursive sub-calls
4. Output `FINAL()` / `FINAL_VAR()` properly
5. Handle infinite context via RLM patterns